In [1]:
from pynq import Overlay
import numpy as np
import time

In [ ]:
class PynqTestDriver:
    def __init__(self, bitfile_path):
        self.hw = Overlay(bitfile_path)
        self.csr = self.hw.csr_0.mmio.array
        self.sram1 = self.hw.SRAM1.mmio.array
        self.sram2 = self.hw.SRAM2.mmio.array
        self.result = np.zeros(192)
        
    def get_output(self, length):
        return self.sram2[0:length]

    def start(self, input_array):
        start_time = time.time()
        """
        Write your own code to start the SRAM controller.
        1. Store the input_array to SRAM1.
        2. "Start" your SRAM controller.
        3. Wait until the controller is "Done"
        4. Load the output result from SRAM2 to self.result.
        """
        ###################################################
        # 1. Store the input_array to SRAM1
        self.sram1[0:len(input_array)] = input_array
        
        # 2. "Start" SRAM controller Data Transfer
        self.csr[1] = 0x1   # Start 신호 전달
        
        # 3. Wait until the controller is "Done"
        while self.csr[0] & 0x1 == 0:   # trigger start
            continue
        # 4. Load the output result from SRAM2 to self.result
        self.result = self.get_output(192)
        ###################################################
        
        end_time = time.time()
        runtime = end_time - start_time
        print(f"Runtime: {runtime*1000:.3f}ms")


    def verify_output(self, answer_path="answer_memory.npy", length=192):
        answer = np.load(answer_path)
        mismatch_count = np.sum(self.result != answer)
        if mismatch_count == 0:
            print("Verification Passed")
        else:
            print("Verification Failed")

In [3]:
driver = PynqTestDriver("sram_controller.bit")

In [4]:
test_input = np.arange(256, dtype=np.int32) + 1
driver.start(test_input)

Runtime: 0.676ms


In [5]:
driver.verify_output()

Verification Passed


In [6]:
print(driver.csr[1])

0


In [ ]:
print(driver.csr[0])